# etm notebook

## vscode shortcuts

- ctrl + =: reset editor group size (make window widths equal)
- cmd + k + s: show shortcuts

- cmd + `:   show/hide terminal
- ctrl + b:  show/hide panel
- shift + cmd + e: switch view to panel and back
- shift + cmd + f: switch to find
- cmd + k, z: zen mode
- shift + cmd + b: open command palette

- cmd + p: file explorer
- cmd + up/down: beginning/end of file
- shift + F12: show uses for cursor item
- F12: goto definition of cursor item
- **F2: rename all instances of cursor variable**
- opt + up/down: move line up/down
- shift + opt + up/down: copy line up/down

- opt + cmd + left/right: previous/next tab



all relevant hashes in dataview

schedule item loop -> get_item_views(item, beg_week, end_week)

get_item_views updates the relevant dataview hashes including the weekly views for the range of weeks using either caches or creating as needed

create_item_views calls the various handlers such as handle_today

# Engage brain before starting engine!!!

## __init__?

Most convenient is to use Item and Repeat classes to process an entry string into tokens and then to create the relevant objects. Tokenization, feedback and validation are done in the Item and Repeat classes.


## Item Class

** See class Item below for example **

### Instance Methods

- relevant() - return the relevant datetime for the item or None

- alerts() - return a list of alerts occurring today for the item or []

- agenda(beg_dt, end_dt) - return a list of agenda rows for the range of dates. Empty list if there are no instances in range of dates. When range includes the current date, include beginbys, pastdues and inbox

- effort(beg_dt, end_dt) - return a list of effort rows for the range of dates. Empty list if there are no instances in range of dates

- ditto for other weekly views

- history() - return the history row for the item

- ditto for other non-weekly views

- weekly_views(beg_wk, end_wk) - return a hash with view keys and lists of corresponding view rows for the range of weeks.

    - weekly view names: agenda, busy, effort, completed

    - data structure for weekly views: view_name -> list of (iso-datetime, view row) tuples where iso-datetime is the integer tuple (year, week number, weekday number, 24-hour hour number, minute number)

## Repeat Class

In [208]:
# NOTE: unwrap to encode and wrap to decode @d entries?
import textwrap
import shutil

def wrap(txt, indent=1, width=shutil.get_terminal_size()[0] - 3):
    para = [x.rstrip() for x in txt.split('\n')]
    tmp = []
    first = True
    for p in para:
        if first:
            initial_indent = ''
            first = False
        else:
            initial_indent = ' ' * indent
        tmp.append(
            textwrap.fill(
                p,
                initial_indent=initial_indent,
                subsequent_indent=' ' * indent,
                width=width - indent - 1,
            )
        )
    return '\n'.join(tmp)

def unwrap(wrapped_text):
    # Split the text into paragraphs
    paragraphs = wrapped_text.split('\n')

    # Remove indentations and join lines within each paragraph
    unwrapped_paragraphs = []
    current_paragraph = []

    for line in paragraphs:
        if line.strip() == '':
            # Paragraph separator
            if current_paragraph:
                unwrapped_paragraphs.append(' '.join(current_paragraph))
                current_paragraph = []
            unwrapped_paragraphs.append('')
        else:
            # Remove leading spaces used for indentation
            current_paragraph.append(line.strip())

    # Add the last paragraph if there is any
    if current_paragraph:
        unwrapped_paragraphs.append(' '.join(current_paragraph))

    # Join the unwrapped paragraphs
    return '\n'.join(unwrapped_paragraphs)

# Example usage
original_text = """\
This is a long string that we want to wrap and then unwrap. We will use the textwrap module to handle the wrapping.
This is a new paragraph that should also be wrapped.
"""
original_text += "  Now is the time for all good men to come to the aid of their country.\n" * 5

# Wrap the text
wrapped_text = wrap(original_text, indent=3, width=50)
print("Wrapped text:")
print(wrapped_text)

# Unwrap the text
unwrapped_text = unwrap(wrapped_text)
print("\nUnwrapped text:")
print(unwrapped_text)

Wrapped text:
This is a long string that we want to wrap and
   then unwrap. We will use the textwrap
   module to handle the wrapping.
   This is a new paragraph that should also be
   wrapped.
     Now is the time for all good men to come
   to the aid of their country.
     Now is the time for all good men to come
   to the aid of their country.
     Now is the time for all good men to come
   to the aid of their country.
     Now is the time for all good men to come
   to the aid of their country.
     Now is the time for all good men to come
   to the aid of their country.


Unwrapped text:
This is a long string that we want to wrap and then unwrap. We will use the textwrap module to handle the wrapping. This is a new paragraph that should also be wrapped. Now is the time for all good men to come to the aid of their country. Now is the time for all good men to come to the aid of their country. Now is the time for all good men to come to the aid of their country. Now is the time fo

In [4]:
from dateutil.rrule import rrule, rruleset, rrulestr, DAILY

ruleset = rrulestr("""
DTSTART:19970902T090000
RRULE:FREQ=DAILY;INTERVAL=10;COUNT=5
RRULE:FREQ=DAILY;INTERVAL=5;COUNT=3
""")

list(ruleset)



[datetime.datetime(1997, 9, 2, 9, 0),
 datetime.datetime(1997, 9, 7, 9, 0),
 datetime.datetime(1997, 9, 12, 9, 0),
 datetime.datetime(1997, 9, 22, 9, 0),
 datetime.datetime(1997, 10, 2, 9, 0),
 datetime.datetime(1997, 10, 12, 9, 0)]

In [23]:
import re
wkd_regex = re.compile(r'(?<![\w-])([+-][1-4])?(MO|TU|WE|TH|FR|SA|SU)(?!\w)')
# wkd_regex = re.compile(r'([+-][1-4])?(MO|TU|WE|TH|FR|SA|SU)')
wkd_str = ("-1MO, +2Tu, WE, -5TH, 1FR, -3SA, 5SU").upper()
matches = wkd_regex.findall(wkd_str)
_ = [f"{x[0]}{x[1]}" for x in matches]
all = [x.strip() for x in wkd_str.split(',')]
bad = [x for x in all if x not in good]
good = _
# for x in matches:
#     arg = f"+{x[0]}" if x[0] and not x[0].startswith('-') else x[0]
#     s = f"{x[0]{x[1]}({arg})" if arg else f"rrule.{x[1]}"
#     good.append(eval(s))

print('\nall:', all, '\nbad:', bad, '\ngood:', good)


all: ['-1MO', '+2TU', 'WE', '-5TH', '1FR', '-3SA', '5SU'] 
bad: ['-5TH', '1FR', '5SU'] 
good: ['-1MO', '+2TU', 'WE', '-3SA']


## Views

- iterate through all items appending relevant 

- agenda (cached?). Create collects agenda rows from all items, sorts and makes tree

# TODO

-  _from_entry_string: for Item, Repeat and Jobs. 

- Clean up Dataview and Item classes - what's actually needed for schedule?

- 

## Unit Tests

In [ ]:
# my_module.py
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

In [ ]:
# pytest actions
# 1) Discover all files matching test_*.py or *_test.py.
# 2) Run all functions and methods prefixed with test_.

# illustrative pytest "test_*.py" file for my_module.py above
# tests/test_my_module.py
import pytest
from my_module import add, subtract

def test_add():
    assert add(1, 2) == 3
    assert add(-1, 1) == 0
    assert add(0, 0) == 0

def test_subtract():
    assert subtract(2, 1) == 1
    assert subtract(2, 2) == 0
    assert subtract(0, 1) == -1

if __name__ == "__main__":
    pytest.main()

In [168]:
import re
from dateutil.rrule import rruleset, rrule, DAILY
from datetime import datetime, timedelta
import pytz

class Item:
    def __init__(self, json_dict=None, input_string=None):
        self.created = self._get_current_timestamp()
        self.itemtype = None
        self.summary = None
        self.start = None
        self.end = None
        self.recurrence = None
        self.rruleset = None
        if json_dict:
            self._init_from_json(json_dict)
        elif input_string:
            self.parse_input(input_string)

    def _get_current_timestamp(self):
        return datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%S")

    def _init_from_json(self, json_dict):
        self.created = json_dict.get("created", self.created)
        self.itemtype = json_dict.get("itemtype")
        self.summary = json_dict.get("summary")
        self.start = self._parse_datetime(json_dict.get("s", "").replace("{T}:", ""))
        self.end = self._parse_datetime(json_dict.get("e", "").replace("{T}:", ""))
        self.recurrence = json_dict.get("r", [{}])[0]

    def parse_input(self, input_string):
        tokens = self._tokenize(input_string)
        self._parse_tokens(tokens)
        self._validate()

    def _tokenize(self, input_string):
        pattern = r'(@\w+ [^@&]+)|(&\w+ \S+)|(^\S+)|(\S[^@&]*)'
        matches = re.findall(pattern, input_string)
        return [match[0] or match[1] or match[2] or match[3] for match in matches if match[0] or match[1] or match[2] or match[3]]

    def _parse_tokens(self, tokens):
        self.itemtype = tokens[0][0]
        summary_tokens = []
        recurrence_attributes = {}
        for token in tokens[1:]:
            if token.startswith('@'):
                break
            summary_tokens.append(token)
        self.summary = ' '.join(summary_tokens)
        for token in tokens[len(summary_tokens) + 1:]:
            if token.startswith('@s'):
                self.start = self._parse_datetime(token[3:].strip())
            elif token.startswith('@e'):
                self.end = self._parse_duration(token[3:].strip())
            elif token.startswith('@r'):
                self.recurrence = self._parse_recurrence(token[3:].strip())
            elif token.startswith('&'):
                attribute, value = self._parse_attribute(token)
                recurrence_attributes[attribute] = value
        if self.recurrence:
            self.recurrence.update(recurrence_attributes)

    def _parse_datetime(self, datetime_str):
        if not datetime_str:
            return None
        try:
            return datetime.strptime(datetime_str, "%Y%m%dT%H%M%S")
        except ValueError:
            try:
                return datetime.strptime(datetime_str, "%Y/%m/%d")
            except ValueError:
                raise ValueError(f"Invalid datetime format: {datetime_str}")

    def _parse_duration(self, duration_str):
        match = re.match(r'(\d+)([dwmy])', duration_str)
        if not match:
            raise ValueError(f"Invalid duration format: {duration_str}")
        value, unit = match.groups()
        if unit == 'd':
            return timedelta(days=int(value))
        elif unit == 'w':
            return timedelta(weeks=int(value))
        elif unit == 'm':
            return timedelta(days=int(value) * 30)
        elif unit == 'y':
            return timedelta(days=int(value) * 365)

    def _parse_recurrence(self, recurrence_str):
        return {"r": recurrence_str}

    def _parse_attribute(self, attribute_str):
        key, value = attribute_str[1:].split()
        if key == "M":
            value = [int(value)]
        elif key == "w":
            value = [f"{value[:1]}TH"]
        return key, value

    def _validate(self):
        if self.itemtype == '*' and not self.start:
            raise ValueError("Events must have a start datetime (@s)")
        if self.recurrence and not self.start:
            raise ValueError("Items with recurrence (@r) must have a start datetime (@s)")

    def to_dict(self):
        data = {
            "created": self.created,
            "itemtype": self.itemtype,
            "summary": self.summary,
        }
        if self.start:
            data["s"] = "{T}:" + self.start.strftime("%Y%m%dT%H%M%S")
        if self.end:
            data["e"] = "{T}:" + (self.start + self.end).strftime("%Y%m%dT%H%M%S")
        if self.recurrence:
            data["r"] = [self.recurrence]
        return data

    def __repr__(self):
        return str(self.to_dict())

# Example usage
json_entry = {
    "created": "{T}:20240712T1052",
    "itemtype": "*",
    "summary": "Thanksgiving",
    "s": "{T}:20101126T0500",
    "r": [
        {
            "r": "y",
            "M": [11],
            "w": ["4TH"]
        }
    ],
    "modified": "{T}:20240712T1054"
}

item_from_json = Item(json_dict=json_entry)
print(item_from_json)

item_from_string = Item(input_string="* Thanksgiving @s 2010/11/26 @r y &M 11 &w 4TH")
print(item_from_string)

{'created': '{T}:20240712T1052', 'itemtype': '*', 'summary': 'Thanksgiving', 's': '{T}:20101126T050000', 'r': [{'r': 'y', 'M': [11], 'w': ['4TH']}]}
{'created': '20240722T124522', 'itemtype': '*', 'summary': 'Thanksgiving ', 's': '{T}:20101126T000000', 'r': [{'r': 'y', 'M': [11], 'w': ['4TH']}]}


In [ ]:
import re
from dateutil.rrule import rruleset, rrule, DAILY
from datetime import datetime, timedelta
import pytz

class Item:
    def __init__(self, input_string=None):
        self.created = self._get_current_timestamp()
        self.itemtype = None
        self.summary = None
        self.start = None
        self.end = None
        self.recurrence = None
        self.rruleset = None
        if input_string:
            self.parse_input(input_string)

    def _get_current_timestamp(self):
        return datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%S")

    def parse_input(self, input_string):
        tokens = self._tokenize(input_string)
        self._parse_tokens(tokens)
        self._validate()

    def _tokenize(self, input_string):
        pattern = r'(@\w+ [^@]+)|(^\S+)|(\S[^@]*)'
        matches = re.findall(pattern, input_string)
        return [match[0] or match[1] or match[2] for match in matches if match[0] or match[1] or match[2]]

    def _parse_tokens(self, tokens):
        self.itemtype = tokens[0][0]
        summary_tokens = []
        for token in tokens[1:]:
            if token.startswith('@'):
                break
            summary_tokens.append(token)
        self.summary = ' '.join(summary_tokens)
        for token in tokens[len(summary_tokens) + 1:]:
            if token.startswith('@s'):
                self.start = self._parse_datetime(token[3:].strip())
            elif token.startswith('@e'):
                self.end = self._parse_duration(token[3:].strip())
            elif token.startswith('@r'):
                self.recurrence = self._parse_recurrence(token[3:].strip())
            # Add additional parsing as needed

    def _parse_datetime(self, datetime_str):
        try:
            return datetime.strptime(datetime_str, "%Y/%m/%d")
        except ValueError:
            raise ValueError(f"Invalid datetime format: {datetime_str}")

    def _parse_duration(self, duration_str):
        match = re.match(r'(\d+)([dwmy])', duration_str)
        if not match:
            raise ValueError(f"Invalid duration format: {duration_str}")
        value, unit = match.groups()
        if unit == 'd':
            return timedelta(days=int(value))
        elif unit == 'w':
            return timedelta(weeks=int(value))
        elif unit == 'm':
            return timedelta(days=int(value) * 30)
        elif unit == 'y':
            return timedelta(days=int(value) * 365)

    def _parse_recurrence(self, recurrence_str):
        # Implement recurrence parsing logic
        pass

    def _validate(self):
        if self.itemtype == '*' and not self.start:
            raise ValueError("Events must have a start datetime (@s)")
        if self.recurrence and not self.start:
            raise ValueError("Items with recurrence (@r) must have a start datetime (@s)")

    def to_dict(self):
        data = {
            "created": self.created,
            "itemtype": self.itemtype,
            "summary": self.summary,
        }
        if self.start:
            data["s"] = self.start.strftime("%Y%m%dT%H%M%S")
        if self.end:
            data["e"] = (self.start + self.end).strftime("%Y%m%dT%H%M%S")
        if self.recurrence:
            data["r"] = self.recurrence
        return data

    def __repr__(self):
        return str(self.to_dict())

# Example usage
item = Item("* carpe diem @s 2024/7/10 @r d")
print(item)

item2 = Item("- ask ChatGPT how to fix my code")
print(item2)

In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @classmethod
    def from_string(cls, date_str):
        year, month, day = map(int, date_str.split('-'))
        return cls(year, month, day)

date1 = Date(2024, 7, 5)
date2 = Date.from_string("2024-07-05")

In [ ]:
class Calculator:
    constants = {
        "pi": 3.14,
        "e": 2.71}

    @classmethod
    def add(cls, a, b):
        return cls.constants.get(a, a) + cls.constants.get(b, b)

    @classmethod
    def subtract(cls, a, b):
        return cls.constants.get(a, a) - cls.constants.get(b, b)

print(Calculator.add(5, 3))
print(Calculator.subtract(5, 3))
print(Calculator.add(5, "pi"))

In [4]:
# FIXME: not working
from datetime import datetime, date, timedelta
from dateutil import rrule
from dateutil.rrule import rruleset, DAILY, rrulestr
from dateutil.tz import gettz
import textwrap

print(f"{dict(a='three', b='two')}")

# def is_naive(dt):
#     return dt.tzinfo is None or dt.tzinfo.utcoffset(dt) is None

# Function to create a string representation of the rruleset
def rruleset_to_string(rruleset_obj):
    parts = []
    # parts.append("rrules:")
    for rule in rruleset_obj._rrule:
        # parts.append(f"{textwrap.fill(str(rule))}")
        parts.append(f"{'\\n'.join(str(rule).split('\n'))}")
    # parts.append("exdates:")
    for exdate in rruleset_obj._exdate:
        parts.append(f"EXDATE:{exdate}")
    # parts.append("rdates:")
    for rdate in rruleset_obj._rdate:
        parts.append(f"RDATE:{rdate}")
    return "\n".join(parts)

# Define the timezone (replace 'America/New_York' with your specific timezone)
pacific = gettz('US/Pacific')
mountain = gettz('America/Denver')
central = gettz('US/Central')
eastern = gettz('America/New_York')
local = gettz()
utc = gettz('UTC')
naive = None

tz = naive
# Define the start date
start_date = datetime(2024,7,20,13,0,0).astimezone().replace(tzinfo=None)


def dt_to_naive(dt):
    return dt if dt.tzinfo is None or dt.tzinfo.utcoffset(dt) is None else dt.replace(tzinfo=None)

no_r = rruleset()
no_r.rdate(start_date)
no_r.rdate(start_date + timedelta(days=5))
print(f"no_r = {rruleset_to_string(no_r)}")
print(f"no_r dates: {list(no_r)}")

rules_lst = []

# Create a recurrence rule for daily events
rule1 = rrule.rrule(freq=DAILY, dtstart=start_date, count=14)
rules_lst.append(str(rule1))
# Create another recurrence rule for specific days (e.g., every 2 days)
rule2 = rrule.rrule(freq=DAILY, interval=2, count=7)
rules_lst.append(str(rule2))
print(f"{rules_lst = }")

# Create an rruleset
rules1 = rruleset()
rules2 = rruleset()
rules12 = rruleset()

# Add the rules to the rruleset
rules1.rrule(rule1)
rules2.rrule(rule2)
rules12.rrule(rule1)
rules12.rrule(rule2)

# Add a specific date to include
plusdates = [datetime(2024, 11, 4, 13, 45, tzinfo=tz), datetime(2024, 11, 5, 15, 15, tzinfo=tz)]
for dt in plusdates:
    dt = dt_to_naive(dt)
    rules1.rdate(dt)  # dt_to_naive(dt)
    rules2.rdate(dt)  # dt_to_naive(dt)
    rules12.rdate(dt)  # dt_to_naive(dt)
    rules_lst.append(dt.strftime("RDATE:%Y%m%dT%H%M%S"))

# Add a specific date to exclude
minusdates = [datetime(2024, 11, 4, 13, 30, tzinfo=tz),]
for dt in minusdates:
    dt = dt_to_naive(dt)
    rules12.exdate(dt)
    rules_lst.append(dt.strftime("EXDATE:%Y%m%dT%H%M%S"))

# Generate the occurrences of the event
occurrences1 = list(rules1)
occurrences2 = list(rules2)
occurrences12 = list(rules12)

# xafter_eg = rules12.xafter(start_date, count=3, inc=True)
# print(f"xafter_eg = {list(after_eg) = }")

# between_eg = rules12.between(datetime.now(), datetime.now() + timedelta(days=14), count=3, inc=True)
# print(f"between_eg = {list(between_eg) = }")

print(f"\nrules:")
print(rruleset_to_string(rules12))
print(f"{rules12.__dict__ = }")

# Print the occurrences
print("\noccurrences from rules1:")
for occurrence in occurrences1:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

print("\noccurrences from rules2:")
for occurrence in occurrences2:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

print("\noccurrences from rules12:")
for occurrence in occurrences12:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

# print("\nlist of string representations of rules:")
# print('\n'.join(rules_lst))

# print("\nfrom list of string representations to new rules:")
# rules_from_str = rrulestr('\n'.join(rules_lst))
# print(rruleset_to_string(rules_from_str))

# occurrences_from_str = list(rules_from_str)
# print("\noccurrences from new rules:")
# for occurrence in occurrences_from_str:
#     print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

{'a': 'three', 'b': 'two'}
no_r = RDATE:2024-07-20 13:00:00
RDATE:2024-07-25 13:00:00
no_r dates: [datetime.datetime(2024, 7, 20, 13, 0), datetime.datetime(2024, 7, 25, 13, 0)]
rules_lst = ['DTSTART:20240720T130000\nRRULE:FREQ=DAILY;COUNT=14', 'DTSTART:20240723T095034\nRRULE:FREQ=DAILY;INTERVAL=2;COUNT=7']

rules:
DTSTART:20240720T130000\nRRULE:FREQ=DAILY;COUNT=14
DTSTART:20240723T095034\nRRULE:FREQ=DAILY;INTERVAL=2;COUNT=7
EXDATE:2024-11-04 13:30:00
RDATE:2024-11-04 13:45:00
RDATE:2024-11-05 15:15:00
rules12.__dict__ = {'_cache': None, '_cache_complete': False, '_len': 23, '_rrule': [<dateutil.rrule.rrule object at 0x107133b60>, <dateutil.rrule.rrule object at 0x1108c8b00>], '_rdate': [datetime.datetime(2024, 11, 4, 13, 45), datetime.datetime(2024, 11, 5, 15, 15)], '_exrule': [], '_exdate': [datetime.datetime(2024, 11, 4, 13, 30)]}

occurrences from rules1:
  Sat 2024-07-20 13:00  
  Sun 2024-07-21 13:00  
  Mon 2024-07-22 13:00  
  Tue 2024-07-23 13:00  
  Wed 2024-07-24 13:00  
  Th

In [1]:
# Create an rruleset
rules = rruleset()

# Add the rules to the rruleset
rules.rrule(rule1)
rules.rrule(rule2)

# Add a specific date to include
plusdates = [datetime(2024, 11, 4, 13, 45, tzinfo=tz), datetime(2024, 11, 5, 15, 15, tzinfo=tz)]
for dt in plusdates:
    dt = dt_to_naive(dt)
    rules.rdate(dt)  # dt_to_naive(dt)
    rules_lst.append(dt.strftime("RDATE:%Y%m%dT%H%M%S"))

# Add a specific date to exclude
minusdates = [datetime(2024, 11, 4, 13, 30, tzinfo=tz),]
for dt in minusdates:
    dt = dt_to_naive(dt)
    rules.exdate(dt)
    rules_lst.append(dt.strftime("EXDATE:%Y%m%dT%H%M%S"))

# Generate the occurrences of the event
occurrences = list(rules)

print(f"\nrules:")
print(rruleset_to_string(rules))

# Print the occurrences
print("\noccurrences from rules:")
for occurrence in occurrences:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

print("\nlist of string representations of rules:")
print('\n'.join(rules_lst))

print("\nfrom list of string representations to new rules:")
rules_from_str = rrulestr('\n'.join(rules_lst))
print(rruleset_to_string(rules_from_str))

occurrences_from_str = list(rules_from_str)
print("\noccurrences from new rules:")
for occurrence in occurrences_from_str:
    print(occurrence.strftime("  %a %Y-%m-%d %H:%M %Z %z"))

NameError: name 'rruleset' is not defined

In [170]:
import re

# Define the regex pattern
pattern = re.compile(r'(?<![\w-])(-?[1-4]?)(MO|TU|WE|TH|FR|SA|SU)(?!\w)')

# Example test string
test_string = "MO, -1TU, 4FR, WE, SA, -3MO, 2WE, -4FR, 5FR, XYZ, -5MO"

# Find all matches
matches = re.findall(pattern, test_string)

def maybe_int(x):
    try:
        return int(x)
    except ValueError:
        return x

good = [f"{maybe_int(x[0])}{x[1]}" for x in matches]
all = [x.strip() for x in test_string.split(',')]
bad = [x for x in all if x not in good]

print("good:", good)
print("all:", all)
print("bad:", bad)

# Print the matches
print(matches)

good: ['MO', '-1TU', '4FR', 'WE', 'SA', '-3MO', '2WE', '-4FR']
all: ['MO', '-1TU', '4FR', 'WE', 'SA', '-3MO', '2WE', '-4FR', '5FR', 'XYZ', '-5MO']
bad: ['5FR', 'XYZ', '-5MO']
[('', 'MO'), ('-1', 'TU'), ('4', 'FR'), ('', 'WE'), ('', 'SA'), ('-3', 'MO'), ('2', 'WE'), ('-4', 'FR')]


In [169]:
# TODO: 'r' can be a dict, entry_string, or rule_string. Need methods to convert one form to another
def item_dict_to_string(item: dict[str, str])->str:
    format_dict = {
        'itemtype': "",
        'summary': " ",
        's': f"\n@s ",
        'e': f" @e ",
        'r': f"\n@r ",
    }

    formatted_string = ""
    for key in format_dict.keys():
        if key in item:
            formatted_string += f"{format_dict[key]}{item[key]}"
    return formatted_string


# Example usage
example = {'itemtype': '*', 'summary': 'Thanksgiving', 's': '20241128T000000', 'r': 'RRULE:FREQ=YEARLY;BYMONTH=11;BYDAY=4TH'}

print(item_dict_to_string(example))

* Thanksgiving
@s 20241128T000000
@r RRULE:FREQ=YEARLY;BYMONTH=11;BYDAY=4TH


# NOTE: Tracking
## Tracking

For undated tasks, add option for adding @o t to add completions to @h instead of simply marking the task as completed by adding an @f entry.

One possibility would be to have @h be a tuple of (number of completions, average timedelta between completions, datetime of last completion)

In [180]:
# NOTE: Tracker and TrackerManager Classes
from typing import List, Dict, Any, Callable, Mapping
from collections import defaultdict
from datetime import datetime, date, timedelta
from dateutil.parser import parse
import traceback
import sys

default_history = [0, timedelta(minutes=0), None]
ZERO = timedelta(minutes=0)

class Tracker:
    _next_id = 1
    default_history = [0, timedelta(minutes=0), None]

    @classmethod
    def format_dt(self, dt: Any) -> str:
        if not isinstance(dt, datetime):
            return ""
        return dt.strftime("%Y-%m-%d %H:%M")

    @classmethod
    def parse_dt(self, dt: str = "") -> datetime:
        # print(f"parsing {dt = }; type(dt) = {type(dt)}")
        if isinstance(dt, datetime):
            return dt
        elif isinstance(dt, str) and dt:
            try:
                # print(f"parsing {dt = }")
                dt = parse(dt)
                # print(f"returning {dt = }")
                return dt
            except Exception as e:
                print(f"Error parsing datetime: {dt}\ne {repr(e)}\n{traceback.format_exc()}", file=sys.stderr, flush=True)
                return None
        else:
            return None

    def __init__(self, name: str, dt: str = "") -> None:
        self.doc_id = Tracker._next_id
        Tracker._next_id += 1
        self.name = name
        self.last_completion = None
        self.last_interval = None
        self.next_expected_completion = None
        self.history = Tracker.default_history

    def record_completion(self, dt):
        self.last_completion = Tracker.parse_dt(dt)
        # print(f"{dt = } -> {self.last_completion = }")
        # print(f"self.last_completion: {Tracker.format_dt(self.last_completion)}")

        num_intervals, average_interval, last_completion = self.history
        if last_completion is None:
            num_intervals = 0
            average_interval = ZERO
            last_completion = self.last_completion
        else:
            self.last_interval = self.last_completion - last_completion
            total = num_intervals * average_interval + self.last_interval
            num_intervals += 1
            average_interval = total / num_intervals

        self.history = [num_intervals, average_interval, self.last_completion]
        self
        self.next_expected_completion = self.last_completion + average_interval if (self.last_completion is not None and self.history[1] is not ZERO) else None

    def get_tracker_data(self):
        # print(f"{self.history = }")
        return f"""{self.name}:
    completion intervals: {self.history[0]}
    average interval: {str(self.history[1])}
    last interval: {str(self.last_interval)}
    last_completion: {Tracker.format_dt(self.last_completion)}
    next_expected_completion: {Tracker.format_dt(self.next_expected_completion)}"""

class TrackerManager:
    def __init__(self) -> None:
        self.trackers = {}

    def add_tracker(self, tracker) -> None:
        doc_id = tracker.doc_id
        self.trackers[doc_id] = tracker

    def record_completion(self, doc_id: int, dt: datetime):
        self.trackers[doc_id].record_completion(dt)
        print(f"""\
{doc_id}: Recorded {dt.strftime('%Y-%m-%d %H:%M')} as a completion  {self.trackers[doc_id].get_tracker_data()}""")

    def get_tracker_data(self, doc_id: int = None):
        if doc_id is None:
            for k, v in self.trackers.items():
                print(f"{k}: {v.get_tracker_data()}")
        elif doc_id in self.trackers:
            print(f"{doc_id}: {self.trackers[doc_id].get_tracker_data()}")

    def list_trackers(self):
        for k, v in self.trackers.items():
            print(f"{k}: {v.name}")

print("creating manager")
manager = TrackerManager()
print("adding tracker for bird feeder")
manager.add_tracker(Tracker("fill bird feeders"))
datetimes = [parse("2024/6/23 5:38p"),  parse("2024/7/6 4:44p"), parse("2024/7/21 4:18p")]
for dt in datetimes:
    manager.record_completion(1, dt)

for name in ['fill water dispenser', 'haircut', 'fill cat feeder']:
    manager.add_tracker(Tracker(name))

manager.record_completion(2, parse("6:35p"))

datetimes = [datetime(2024, 7, 20, 13, 45), datetime(2024, 7, 22, 15, 15), datetime(2024, 7, 25, 9, 45), datetime(2024, 7, 26, 18, 15)]
print("adding tracker for test")
manager.add_tracker(Tracker("test tracker"))
for dt in datetimes:
    manager.record_completion(5, dt)

print("\nall manager data:")
manager.get_tracker_data()
# manager.list_trackers()

creating manager
adding tracker for bird feeder
1: Recorded 2024-06-23 17:38 as a completion  fill bird feeders:
    completion intervals: 0
    average interval: 0:00:00
    last interval: None
    last_completion: 2024-06-23 17:38
    next_expected_completion: 
1: Recorded 2024-07-06 16:44 as a completion  fill bird feeders:
    completion intervals: 1
    average interval: 12 days, 23:06:00
    last interval: 12 days, 23:06:00
    last_completion: 2024-07-06 16:44
    next_expected_completion: 2024-07-19 15:50
1: Recorded 2024-07-21 16:18 as a completion  fill bird feeders:
    completion intervals: 2
    average interval: 13 days, 23:20:00
    last interval: 14 days, 23:34:00
    last_completion: 2024-07-21 16:18
    next_expected_completion: 2024-08-04 15:38
2: Recorded 2024-07-22 18:35 as a completion  fill water dispenser:
    completion intervals: 0
    average interval: 0:00:00
    last interval: None
    last_completion: 2024-07-22 18:35
    next_expected_completion: 
adding 

In [87]:
from collections import defaultdict
from datetime import date, timedelta
import textwrap
import shutil
import sys

def wrap(txt, indent=3, width=shutil.get_terminal_size()[0] - 1):
    """
    Wrap text to terminal width using indent spaces before each line.
    >>> txt = "Now is the time for all good men to come to the aid of their country. " * 5
    >>> res = wrap(txt, 4, 60)
    >>> print(res)
    Now is the time for all good men to come to the aid of
        their country. Now is the time for all good men to
        come to the aid of their country. Now is the time
        for all good men to come to the aid of their
        country. Now is the time for all good men to come
        to the aid of their country. Now is the time for
        all good men to come to the aid of their country.
    """
    para = [x.rstrip() for x in txt.split('\n')]
    tmp = []
    first = True
    for p in para:
        if first:
            initial_indent = ''
            first = False
        else:
            initial_indent = ' ' * indent
        tmp.append(
            textwrap.fill(
                p,
                initial_indent=initial_indent,
                subsequent_indent=' ' * indent,
                width=width - indent - 1,
            )
        )
    return '\n'.join(tmp)

def print_wrapped(txt):
    print(wrap(txt))

class Item:
    def __init__(self, doc_id, name, item_type, item_date=None, completion_date=None):
        self.doc_id = doc_id
        self.item_type = item_type
        self.name = name
        self.item_date = item_date
        self.completion_date = completion_date
        self.is_completed = (
            self.item_type == 'task' and
            self.completion_date is not None
            )
        # Add other properties as needed
        # Add other properties as needed

    def get_views_and_rows(self):
        views_and_rows = {}

        # Example logic for adding to agenda view if there is an event date
        if self.item_date:
            YrWk = self.item_date.isocalendar()[:2]
        elif self.completion_date:
            YrWk = self.completion_date.isocalendar()[:2]
        else:
            YrWk = None

        if self.item_type == 'event':
            if self.item_date:
                views_and_rows[(YrWk, 'agenda')] = [(self.item_date, self.name)]

        # Example logic for adding to tasks view if there is a due date and it's not completed
        # Example logic for adding to completed view if it is completed
        elif self.item_type == 'task':
            if self.completion_date:
                views_and_rows[(YrWk, 'agenda')] = [(self.completion_date, self.name)]
            elif self.item_date:
                views_and_rows[(YrWk, 'agenda')] = [(self.item_date, self.name)]
            else:
                views_and_rows['tasks'] = [(None, self.name)]


        # Add other views and logic as needed based on the properties of the item
        # print(f"{views_and_rows = }")
        return views_and_rows


from collections import defaultdict
from datetime import date

class ReminderManager:
    def __init__(self):
        self.doc_view_data = {}  # Primary structure: dict[doc_id, dict[view, list[row]]]
        self.view_doc_data = defaultdict(lambda: defaultdict(list))  # Secondary index: dict[view, dict[doc_id, list[row]])
        self.view_cache = {}  # Cache for views
        self.doc_view_contribution = defaultdict(set)  # Tracks views each doc_id contributes to

    def add_or_update_reminder(self, item):
        doc_id = item.doc_id
        new_views_and_rows = item.get_views_and_rows()

        # Invalidate cache for views that will be affected by this doc_id
        self.invalidate_cache_for_doc(doc_id)

        # Update the primary structure
        self.doc_view_data[doc_id] = new_views_and_rows

        # Update the secondary index
        for view, rows in new_views_and_rows.items():
            self.view_doc_data[view][doc_id] = rows
            self.doc_view_contribution[doc_id].add(view)

    def get_view_data(self, view):
        # Check if the view is in the cache
        if view in self.view_cache:
            return self.view_cache[view]

        # Retrieve data for a specific view
        view_data = dict(self.view_doc_data[view])

        # Cache the view data
        self.view_cache[view] = view_data
        return view_data

    def get_reminder_data(self, doc_id):
        # Retrieve data for a specific reminder
        return self.doc_view_data.get(doc_id, {})

    def remove_reminder(self, doc_id):
        # Invalidate cache for views that will be affected by this doc_id
        self.invalidate_cache_for_doc(doc_id)

        # Remove reminder from primary structure
        if doc_id in self.doc_view_data:
            views_and_rows = self.doc_view_data.pop(doc_id)
            # Remove from secondary index
            for view in views_and_rows:
                if doc_id in self.view_doc_data[view]:
                    del self.view_doc_data[view][doc_id]

            # Remove doc_id from contribution tracking
            if doc_id in self.doc_view_contribution:
                del self.doc_view_contribution[doc_id]

    def invalidate_cache_for_doc(self, doc_id):
        # Invalidate cache entries for views affected by this doc_id
        if doc_id in self.doc_view_contribution:
            for view in self.doc_view_contribution[doc_id]:
                if view in self.view_cache:
                    del self.view_cache[view]

# Example usage
manager = ReminderManager()

# Create some reminders (Item instances)
item1 = Item(1, "Independence Day", "event", item_date=date(2024, 7, 4))
item2 = Item(2, "Dated", "task", item_date=date(2024, 7, 10))
item3 = Item(3, "UnDated", "task")
item4 = Item(4, "Undated and Completed", "task", completion_date=date(2024, 7, 10))
item5 = Item(5, "Dated and Completed", "task", item_date=date(2024, 7, 4), completion_date=date(2024, 7, 6))

# Add or update reminders
manager.add_or_update_reminder(item1)
manager.add_or_update_reminder(item2)
manager.add_or_update_reminder(item3)
manager.add_or_update_reminder(item4)
manager.add_or_update_reminder(item5)

# Query view data
view_data_agenda = manager.get_view_data(((2024, 27), "agenda"))
print_wrapped(f"Agenda View Data for 2024-27:\n{view_data_agenda}")
view_data_agenda = manager.get_view_data(((2024, 28), "agenda"))
print_wrapped(f"Agenda View Data for 2024-28:\n{view_data_agenda}")

view_data_tasks = manager.get_view_data("tasks")
print_wrapped(f"Tasks View Data:\n{view_data_tasks}")

# Query reminder data
reminder_data_1 = manager.get_reminder_data(1)
print_wrapped(f"Reminder 1 Data:\n{reminder_data_1}")

print_wrapped(f"view_cache:\n{manager.view_cache}")

# Remove a reminder
manager.remove_reminder(5)
print("After removal of item 5")
view_data_agenda = manager.get_view_data(((2024, 27), "agenda"))
print_wrapped(f"Agenda View Data for 2024-27:\n {view_data_agenda}")
view_data_agenda = manager.get_view_data(((2024, 28), "agenda"))
print_wrapped(f"Agenda View Data for 2024-28:\n{view_data_agenda}")

Agenda View Data for 2024-27:
   {1: [(datetime.date(2024, 7, 4), 'Independence Day')], 5:
   [(datetime.date(2024, 7, 6), 'Dated and Completed')]}
Agenda View Data for 2024-28:
   {2: [(datetime.date(2024, 7, 10), 'Dated')], 4: [(datetime.date(2024, 7,
   10), 'Undated and Completed')]}
Tasks View Data:
   {3: [(None, 'UnDated')]}
Reminder 1 Data:
   {((2024, 27), 'agenda'): [(datetime.date(2024, 7, 4), 'Independence
   Day')]}
view_cache:
   {((2024, 27), 'agenda'): {1: [(datetime.date(2024, 7, 4), 'Independence
   Day')], 5: [(datetime.date(2024, 7, 6), 'Dated and Completed')]},
   ((2024, 28), 'agenda'): {2: [(datetime.date(2024, 7, 10), 'Dated')], 4:
   [(datetime.date(2024, 7, 10), 'Undated and Completed')]}, 'tasks': {3:
   [(None, 'UnDated')]}}
After removal of item 5
Agenda View Data for 2024-27:
    {1: [(datetime.date(2024, 7, 4), 'Independence Day')]}
Agenda View Data for 2024-28:
   {2: [(datetime.date(2024, 7, 10), 'Dated')], 4: [(datetime.date(2024, 7,
   10), 'Undated a

In [177]:
import re

def get_int_and_str(s):
    # Define a regular expression to match an optional sign followed by digits
    weekdays = ['MO', 'TU', 'WE', 'TH', 'FR', 'SA', 'SU']
    weekday_str = ','.join(weekdays)
    match = re.match(r'^([+-]?\d*)(.*)$', s)
    if match:
        integer_part = match.group(1)
        string_part = match.group(2)
        # Convert integer_part to an integer if it's not empty, otherwise None
        integer_part = integer_part if integer_part else None
        # problems = []
        # if string_part not in weekdays:
        #     problems.append(f"'{string_part}' is not a valid weekday from  {weekday_str}")


        return integer_part, string_part
    return None, s  # Default case if no match is found

# Example usage
examples = ['1MO', '-2T', 'S', '+1FD', '-3WD', '']

for example in examples:
    integer_part, string_part = get_int_and_str(example)
    print(f"Original: {example} -> Integer part: {integer_part} (type: {type(integer_part)}), String part: {string_part}")

Original: 1MO -> Integer part: 1 (type: <class 'str'>), String part: MO
Original: -2T -> Integer part: -2 (type: <class 'str'>), String part: T
Original: S -> Integer part: None (type: <class 'NoneType'>), String part: S
Original: +1FD -> Integer part: +1 (type: <class 'str'>), String part: FD
Original: -3WD -> Integer part: -3 (type: <class 'str'>), String part: WD
Original:  -> Integer part: None (type: <class 'NoneType'>), String part: 


In [196]:
import re
import random
from etm.item import wrap

def _tokenize(self, obj):
    pattern = r'(@\w+ [^@&]+)|(&\w+ \S+)|(^\S+)|(\S[^@&]*)'
    matches = re.finditer(pattern, obj)

    tokens_with_positions = []

    for match in matches:
        # Get the matched token
        token = match.group(0)
        # Get the start and end positions
        start_pos = match.start()
        end_pos = match.end()
        # Append the token and its positions as a tuple
        tokens_with_positions.append((token, start_pos, end_pos))

    return tokens_with_positions

# Example usage
example_string = "* Thanksgiving @s 2010/11/26 @r y &M 11 &w 4TH"

# Assuming the method is part of a class, for demonstration
class TokenClass:

    def __init__(self, entry: str = ""):
        self.entry = ""
        self.tokens = []
        if entry:
            self.process_entry(entry)

    def process_entry(self, entry: str):
        self.entry = entry
        pattern = r'(@\w+ [^@&]+)|(&\w+ \S+)|(^\S+)|(\S[^@&]*)'
        matches = re.finditer(pattern, self.entry)
        tokens_with_positions = []
        for match in matches:
            # Get the matched token
            token = match.group(0)
            # Get the start and end positions
            start_pos = match.start()
            end_pos = match.end()
            # Append the token and its positions as a tuple
            tokens_with_positions.append((token, start_pos, end_pos))
        # print(f"{tokens_with_positions =}")
        self.tokens = tokens_with_positions

    def get_token_at_cursor(self, cursor_pos):
        for token, start_pos, end_pos in self.tokens:
            if start_pos <= cursor_pos < end_pos:
                return token, start_pos, end_pos
        return None, None, None  # No token found at the cursor position


# Instantiate the class and call the method
token_instance = TokenClass(example_string)
# tokens_with_positions = token_instance._tokenize(example_string)
print(f"{token_instance.entry = }")
print(f"token_instance.tokens:")
for token in token_instance.tokens:
    print('  ', token)
cursor_pos = random.choice(range(len(example_string)))
print(f"token at cursor_pos {cursor_pos} = {token_instance.get_token_at_cursor(cursor_pos)}")


token_instance.entry = '* Thanksgiving @s 2010/11/26 @r y &M 11 &w 4TH'
token_instance.tokens:
   ('*', 0, 1)
   ('Thanksgiving ', 2, 15)
   ('@s 2010/11/26 ', 15, 29)
   ('@r y ', 29, 34)
   ('&M 11', 34, 39)
   ('&w 4TH', 40, 46)
token at cursor_pos 27 = ('@s 2010/11/26 ', 15, 29)
